
# Deepfake Detection Challenge Baseline


- **ImageNet pretrained backbone 1개**
- **GAN 기반 얼굴 데이터셋 1개** 사용
- 최종적으로 `submission_baseline.csv` 생성

---

## 목차

1. 환경 설정  
2. 경로 및 하이퍼파라미터 설정  
3. 대회 데이터 압축 해제  
4. 학습 데이터 다운로드 및 로드  
5. 학습/검증 데이터셋 구성  
6. 모델 정의  
7. 학습  
8. 검증  
9. 테스트 추론 및 제출 파일 생성  

---

## 설계

- backbone: `torchvision`의 `ResNet18`
- backbone 대부분 **freeze**
- 외부 데이터는 **GAN 기반 얼굴 데이터셋 1개만** 사용
- 학습 데이터는 전체를 다 쓰지 않고 **작은 subset만 사용**




## 학습 데이터셋

이 베이스라인은 **Kaggle의 `xhlulu/140k-real-and-fake-faces`** 데이터셋을 사용합니다.

- 총 **140,000장**
- **real 70,000장 / fake 70,000장**
- fake 이미지는 **StyleGAN 기반 GAN 얼굴**로 구성되어 있습니다.

이 노트북에서는 라벨을 다음과 같이 사용합니다.

- **real = 0**
- **fake = 1**

즉, `real/` 폴더 이미지는 모두 `0`, `fake/` 폴더 이미지는 모두 `1`로 사용합니다.


In [32]:
# ============================================================
# 1. 필수 패키지 설치
# ============================================================

!pip install timm --quiet 


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import roc_auc_score

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 299
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 15
LR = 1e-4

COMP_DIR = "/kaggle/input/datasets/seoyeon056/competition"
TEST_DIR = os.path.join(COMP_DIR, "test")
FF_DIR = "/kaggle/input/datasets/greatgamedota/faceforensics/cropped_images"

import kagglehub
FACES_DIR = Path(kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces"))

print("DEVICE:", DEVICE)


DEVICE: cuda


In [3]:
# real 이미지: 140k 데이터셋의 real 폴더
real_root = None
for sub in FACES_DIR.rglob("*"):
    if sub.is_dir() and sub.name.lower() == "real":
        real_root = sub
        break

assert real_root is not None, "real 폴더를 찾지 못했습니다."
print("real 폴더:", real_root)

real_paths = list(real_root.glob("*.jpg")) + list(real_root.glob("*.png"))
print("real 이미지 수:", len(real_paths))

# fake 이미지: FaceForensics cropped_images
ff_root = Path(FF_DIR)
fake_paths = []
for folder in ff_root.iterdir():
    if folder.is_dir():
        fake_paths.extend(list(folder.glob("*.png")) + list(folder.glob("*.jpg")))

print("fake 이미지 수:", len(fake_paths))

real 폴더: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid/real
real 이미지 수: 10000
fake 이미지 수: 20351


In [10]:
# 데이터셋 분할
VALID_EXTS = {".jpg", ".jpeg", ".png"}
real_paths = [p for p in FACES_DIR.rglob("*") if p.suffix.lower() in VALID_EXTS and p.parent.name.lower() == "real"]
fake_paths = [p for p in FACES_DIR.rglob("*") if p.suffix.lower() in VALID_EXTS and p.parent.name.lower() == "fake"]
random.Random(SEED).shuffle(real_paths)
random.Random(SEED).shuffle(fake_paths)

N_TRAIN = 50000
N_VAL = 2000

train_real = real_paths[:N_TRAIN]
val_real   = real_paths[N_TRAIN:N_TRAIN + N_VAL]

train_fake = fake_paths[:N_TRAIN]
val_fake   = fake_paths[N_TRAIN:N_TRAIN + N_VAL]

print(f"train: real {len(train_real)}, fake {len(train_fake)}")
print(f"val:   real {len(val_real)},  fake {len(val_fake)}")

train: real 20000, fake 20000
val:   real 2000,  fake 2000


In [11]:
#Dataset, Transform
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, real_paths, fake_paths, transform=None):
        self.samples = [(p, 0) for p in real_paths] + [(p, 1) for p in fake_paths]
        random.shuffle(self.samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(float(label), dtype=torch.float32)

train_dataset = FaceDataset(train_real, train_fake, transform=train_tfms)
val_dataset   = FaceDataset(val_real,   val_fake,   transform=valid_tfms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("train:", len(train_dataset), "/ val:", len(val_dataset))

train: 40000 / val: 4000


In [12]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=1)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [14]:
@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy().tolist()
        all_probs.extend(probs)
        all_labels.extend(y.numpy().tolist())
    return roc_auc_score(all_labels, all_probs)

best_auc = 0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x).squeeze(1)
        y_smooth = y * 0.9 + 0.05
        loss = criterion(logits, y_smooth)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    val_auc = evaluate_auc(model, val_loader)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✓ Best model saved!")

    print(f"[Epoch {epoch+1}] loss={train_loss:.4f} | val_auc={val_auc:.4f}")

print(f"\n최고 val_auc: {best_auc:.4f}")

Epoch 1/10:   0%|          | 0/1250 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))

class TestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root_dir / f"{row['id']}.png").convert("RGB")
        if self.transform:
            img = self.transform(img)
        return row["id"], img

test_dataset = TestDataset(test_df, TEST_DIR, transform=valid_tfms)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

ids, probs = [], []
with torch.no_grad():
    for batch_ids, x in tqdm(test_loader, desc="Predicting"):
        x = x.to(DEVICE)
        logits = model(x).squeeze(1)
        batch_probs = torch.sigmoid(logits).cpu().numpy().tolist()
        ids.extend(list(batch_ids))
        probs.extend(batch_probs)

submission = pd.DataFrame({"id": ids, "score": probs})
submission = sample_sub[["id"]].merge(submission, on="id", how="left")
submission.to_csv("submission.csv", index=False)
print(submission["score"].describe())
print("저장 완료!")
submission.head()

Predicting:   0%|          | 0/10 [00:00<?, ?it/s]

count    300.000000
mean       0.054645
std        0.013931
min        0.023447
25%        0.045374
50%        0.053321
75%        0.061444
max        0.140645
Name: score, dtype: float64
저장 완료!


,id,score
0,test001,0.055640
1,test002,0.046962
2,test003,0.046144
3,test004,0.055936
4,test005,0.044266


In [16]:
submission["score"] = 1 - submission["score"]
submission.to_csv("submission.csv", index=False)
print(submission["score"].describe())


count    300.000000
mean       0.945355
std        0.013931
min        0.859355
25%        0.938556
50%        0.946679
75%        0.954626
max        0.976553
Name: score, dtype: float64
